# BERT Sentiment Classification — Baseline vs. Class-Weighted (RQ2)

Runs `src/train.py` + `src/evaluate.py` twice — once with `use_class_weights: false` (baseline) and once with `use_class_weights: true` (RQ2 class-balancing experiment) — then compares the neutral-class F1/recall between the two.

**Design note:** this notebook's own Jupyter kernel is never assumed to have any project dependency (torch, transformers, pandas, etc.) installed, and nothing is ever installed into it. Every project-specific command runs through a dedicated, fully isolated virtual environment this notebook creates under your home directory (`~/.amazon_sentiment_venv`), using only Python's standard-library `venv` module -- no conda/anaconda needed, and no dependency on write access to whatever kernel environment you happen to be running under. The venv is deliberately isolated from the kernel's own site-packages (not `--system-site-packages`), so `pip install` always installs a correct, working copy of everything (including a CUDA-enabled torch) rather than silently reusing whatever -- possibly broken or CPU-only -- version happens to already be installed in a shared/managed environment. Just run all cells top to bottom.

In [ ]:
# This notebook lives at experiments/bert_class_balancing.ipynb, one
# level below the repo root, and Jupyter starts a notebook's cwd at
# its own directory -- so cwd here is .../experiments, not the repo
# root. Every cell after this one uses paths relative to the repo
# root, so cd up first.
import os

if not os.path.isfile("configs/model_config.yaml") and os.path.isfile("../configs/model_config.yaml"):
    %cd ..

# Assumes the repo is already cloned and this notebook is being run
# from inside it (e.g. a persistent GPU machine/cluster you manage
# yourself). Just makes sure you're on the right branch and up to date.
BRANCH = "main"

!git checkout {BRANCH}

# A previous run of this notebook may have left configs/model_config.yaml
# locally modified (see set_class_weights below) -- stash it first so
# `git pull` can't silently fail/conflict against an uncommitted change.
status = !git status --porcelain configs/model_config.yaml
if status:
    print("Local changes to configs/model_config.yaml detected -- stashing before pull.")
    !git stash push -- configs/model_config.yaml

!git pull
!pwd && ls

In [10]:
# Creates (or reuses) a private virtual environment under your home
# directory, using only the stdlib `venv` module -- no conda/anaconda,
# no --user installs, no write access to the kernel's own environment
# needed. Deliberately NOT using system_site_packages=True: on a
# managed cluster, whatever's already installed in the kernel's own
# environment (e.g. a CPU-only or otherwise broken torch build) would
# get silently reused by pip instead of installing a correct one, which
# is exactly what happened here (`pip show torch` pointed at
# /opt/miniconda/envs/.../site-packages instead of this venv). A fully
# isolated venv guarantees pip installs its own copy of everything.
import os
import shutil
import sys
import venv

VENV_DIR = os.path.expanduser("~/.amazon_sentiment_venv")
VENV_PYTHON = os.path.join(VENV_DIR, "bin", "python")

# If a previous run created this venv with system_site_packages=True
# (the earlier, broken version of this cell), remove it so it gets
# rebuilt fully isolated below.
pyvenv_cfg = os.path.join(VENV_DIR, "pyvenv.cfg")
if os.path.isfile(pyvenv_cfg) and "include-system-site-packages = true" in open(pyvenv_cfg).read().lower():
    print(f"Removing old system-site-packages venv at {VENV_DIR} so it can be rebuilt isolated...")
    shutil.rmtree(VENV_DIR)

if os.path.isfile(VENV_PYTHON):
    print(f"Reusing existing venv at {VENV_DIR}")
else:
    print(f"Creating venv at {VENV_DIR} (based on {sys.executable})...")
    venv.create(VENV_DIR, system_site_packages=False, with_pip=True)
    print("Done.")

print("Using venv Python:", VENV_PYTHON)

Reusing existing venv at /home/furusawa.y/.amazon_sentiment_venv
Using venv Python: /home/furusawa.y/.amazon_sentiment_venv/bin/python


In [11]:
# The default (unpinned) PyPI `torch` wheel bundles whatever CUDA
# runtime is current as of that torch release, which can be newer than
# what this node's NVIDIA driver supports -- exactly what happened here
# (driver reports a max supported CUDA of 12.3 via `nvidia-smi`, but the
# default torch wheel needed something newer, causing "CUDA
# initialization: The NVIDIA driver on your system is too old").
# --force-reinstall is required here: if an earlier run of this
# notebook already installed the (wrong) default-index torch into this
# venv, a plain `pip install torch --index-url ...` with no version
# pin sees torch as "already satisfied" and silently does nothing --
# force-reinstall guarantees the cu121 build actually replaces it. 12.1
# is safely within what a driver supporting up to 12.3 can run.
!{VENV_PYTHON} -m pip install -q --force-reinstall torch --index-url https://download.pytorch.org/whl/cu121
!{VENV_PYTHON} -m pip install -q -r requirements.txt


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: /home/furusawa.y/.amazon_sentiment_venv/bin/python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: /home/furusawa.y/.amazon_sentiment_venv/bin/python -m pip install --upgrade pip


In [12]:
!{VENV_PYTHON} -c "import torch; print('CUDA available:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none -- attach/select a GPU before continuing')"

CUDA available: True
GPU: Tesla V100-SXM2-32GB


## 1. Baseline run (`use_class_weights: false`)

This is already the default in `configs/model_config.yaml`'s `bert.training` section, but the next cell sets it explicitly so this notebook is safe to re-run from the top.

In [ ]:
import re

CONFIG_PATH = "configs/model_config.yaml"


def set_class_weights(enabled: bool) -> None:
    """Flip use_class_weights via a targeted line replacement rather than
    a full yaml.safe_load/safe_dump round-trip, which would silently
    strip every comment in the file (including the RQ2 explanation right
    above this same setting)."""
    text = open(CONFIG_PATH).read()
    new_text, count = re.subn(
        r"^(\s*use_class_weights:[ \t]*)(true|false)[ \t]*$",
        lambda m: f"{m.group(1)}{'true' if enabled else 'false'}",
        text,
        count=1,
        flags=re.MULTILINE,
    )
    if count == 0:
        raise RuntimeError(
            f"Could not find a 'use_class_weights: true|false' line in {CONFIG_PATH}"
        )
    open(CONFIG_PATH, "w").write(new_text)
    print(f"use_class_weights set to {enabled}")


set_class_weights(False)

In [14]:
!{VENV_PYTHON} src/train.py

Loading train split from data/processed/train.csv...
Loading validation split from data/processed/validation.csv...
Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 1194.66it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/arch

In [15]:
!{VENV_PYTHON} src/evaluate.py

Loading test split from data/processed/test.csv...
Evaluating model from models/bert_sentiment_baseline on 7,239 test samples...
Loading weights: 100%|█████████████████████| 201/201 [00:00<00:00, 2155.73it/s]

Accuracy:         0.9048
Precision Macro:  0.7477
Recall Macro:     0.7452
F1 Macro:         0.7464
F1 Weighted:      0.9044

Per-class precision: {'negative': 0.8486370157819225, 'neutral': 0.4340425531914894, 'positive': 0.9605581395348837}
Per-class recall:    {'negative': 0.8510791366906475, 'neutral': 0.422360248447205, 'positive': 0.9621692135669028}
Per-class F1:        {'negative': 0.8498563218390804, 'neutral': 0.4281217208814271, 'positive': 0.9613630015827204}

              precision    recall  f1-score   support

    negative       0.85      0.85      0.85      1390
     neutral       0.43      0.42      0.43       483
    positive       0.96      0.96      0.96      5366

    accuracy                           0.90      7239
   macro avg       0.75      0.75      0.

## 2. Weighted run (`use_class_weights: true`)

In [16]:
set_class_weights(True)

use_class_weights set to True


In [17]:
!{VENV_PYTHON} src/train.py

Loading train split from data/processed/train.csv...
Loading validation split from data/processed/validation.csv...
Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 9256.59it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/arch

In [18]:
!{VENV_PYTHON} src/evaluate.py

Loading test split from data/processed/test.csv...
Evaluating model from models/bert_sentiment_weighted on 7,239 test samples...
Loading weights: 100%|█████████████████████| 201/201 [00:00<00:00, 2194.30it/s]

Accuracy:         0.9050
Precision Macro:  0.7488
Recall Macro:     0.7674
F1 Macro:         0.7576
F1 Weighted:      0.9071

Per-class precision: {'negative': 0.8426573426573427, 'neutral': 0.4338919925512104, 'positive': 0.9698406676783005}
Per-class recall:    {'negative': 0.8669064748201439, 'neutral': 0.4824016563146998, 'positive': 0.9528512858740216}
Per-class F1:        {'negative': 0.8546099290780141, 'neutral': 0.4568627450980392, 'positive': 0.9612709155856364}

              precision    recall  f1-score   support

    negative       0.84      0.87      0.85      1390
     neutral       0.43      0.48      0.46       483
    positive       0.97      0.95      0.96      5366

    accuracy                           0.90      7239
   macro avg       0.75      0.77      0

## 3. Compare baseline vs. weighted (RQ2 answer)

Reads the structured JSON results both runs saved to `outputs/` and compares them directly — the neutral-class row is the one RQ2 cares about. A `NaN` for a metric means that class was never predicted at all (undefined), which is distinct from a real `0.0` (predicted, but always wrong). This cell only uses stdlib (`json`), consistent with keeping the notebook's own kernel dependency-free.

In [ ]:
import json

with open("outputs/bert_evaluation_baseline.json") as f:
    baseline = json.load(f)

with open("outputs/bert_evaluation_weighted.json") as f:
    weighted = json.load(f)

print(f"{'Metric':<24}{'Baseline':>12}{'Weighted':>12}{'Delta':>12}")
print("-" * 60)


def row(label, b, w):
    print(f"{label:<24}{b:>12.4f}{w:>12.4f}{w - b:>+12.4f}")


row("Accuracy", baseline["accuracy"], weighted["accuracy"])
row("Precision Macro", baseline["precision_macro"], weighted["precision_macro"])
row("Recall Macro", baseline["recall_macro"], weighted["recall_macro"])
row("F1 Macro", baseline["f1_macro"], weighted["f1_macro"])
row("F1 Weighted", baseline["f1_weighted"], weighted["f1_weighted"])
print()
for sentiment in ["negative", "neutral", "positive"]:
    row(
        f"Precision ({sentiment})",
        baseline["precision_per_class"][sentiment],
        weighted["precision_per_class"][sentiment],
    )
    row(
        f"Recall ({sentiment})",
        baseline["recall_per_class"][sentiment],
        weighted["recall_per_class"][sentiment],
    )
    row(
        f"F1 ({sentiment})",
        baseline["f1_per_class"][sentiment],
        weighted["f1_per_class"][sentiment],
    )

print()
neutral_f1_delta = weighted["f1_per_class"]["neutral"] - baseline["f1_per_class"]["neutral"]
neutral_recall_delta = weighted["recall_per_class"]["neutral"] - baseline["recall_per_class"]["neutral"]
print(f"Neutral-class F1 delta:     {neutral_f1_delta:+.4f}")
print(f"Neutral-class recall delta: {neutral_recall_delta:+.4f}")
if neutral_f1_delta > 0:
    print(f"RQ2: class weighting IMPROVED neutral-class F1 by {neutral_f1_delta:+.4f}")
elif neutral_f1_delta < 0:
    print(f"RQ2: class weighting HURT neutral-class F1 by {neutral_f1_delta:+.4f}")
else:
    print("RQ2: class weighting made no difference to neutral-class F1")

In [20]:
# Plain stdlib formatting instead of pandas -- keeps this cell dependency-free too.
labels = ["negative", "neutral", "positive"]


def print_confusion_matrix(title, cm):
    print(title)
    print(" " * 12 + "".join(f"{l:>10}" for l in labels))
    for true_label, row_vals in zip(labels, cm):
        print(f"{true_label:>12}" + "".join(f"{v:>10}" for v in row_vals))


print_confusion_matrix("Baseline confusion matrix (rows=true, cols=predicted):", baseline["confusion_matrix"])
print()
print_confusion_matrix("Weighted confusion matrix (rows=true, cols=predicted):", weighted["confusion_matrix"])

Baseline confusion matrix (rows=true, cols=predicted):
              negative   neutral  positive
    negative      1183       130        77
     neutral       144       204       135
    positive        67       136      5163

Weighted confusion matrix (rows=true, cols=predicted):
              negative   neutral  positive
    negative      1205       128        57
     neutral       148       233       102
    positive        77       176      5113


In [ ]:
# !git config user.email "you@example.com"
# !git config user.name "Your Name"
# !git add outputs/
# !git commit -m "Add baseline/weighted BERT evaluation results"
# !git push https://<github-token>@github.com/Sapna-Patel1/amazon-electronics-sentiment-analysis.git {BRANCH}